<!-- Curated copy -->
> **Curated copy.** This notebook is taken verbatim from the BTech-thesis working archive; only
> cell *outputs* have been cleared and machine-specific absolute paths (`C:\\...`, `D:\\...`,
> `F:\\...`) have been rewritten to repository-relative `runs/...` paths. No scientific logic,
> equation, hyper-parameter or architecture has been modified. Place regenerated
> `dataset_run_*` folders under a `runs/` directory next to this notebook (or edit the paths).
> The figures this notebook originally produced are preserved in the sibling `figures/` folder.


In [ ]:
import os, json,ast, math, glob, time
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm
import glob
import pandas as pd
from sklearn.decomposition import PCA

### DATASET GENERATOR and ML PARAMETERS


In [ ]:
N        = 40          # grid points in x,y (Q, T, f are NxN)
Lx, Ly   = 0.05, 0.05
rho, cp, k, L_lat = 800.0, 2000.0, 0.2, 2e5
T_m      = 330.0
T_init   = 300.0
T_bound  = 330.0
t_end    = 4000.0
save_times = (100.0, 250.0, 400.0, 600.0, 1000.0, 1200.0, 1500.0, 1800.0, 2100.0)
cfl      = 0.45

# Heat-source GP
q_scale        = 1e5     # W/m^3
Q_length_scale = 0.18
Q_sigma        = 1.0

# Boundary Conditions
mu_max=0.8
sigma_max=0.7      
amp_max=80

# Dataset sizes
NUM_CASES      = 200.0      # total function-realizations / cases
TRAIN_FRAC     = 0.8
VAL_FRAC       = 0.1      # test is the remainder

# DeepONet sampling
sensor_mode    = "full"   # "full" (flatten full NxN) or "downsample"
S_down         = 40       # used if sensor_mode="downsample" (S_down x S_down grid)
n_time_bc      = len(save_times)
S_down_BC      = N

points_per_case_per_time = N**2  # # of (x,y) points sampled per time snapshot

# Boundary configuration control
only_lr_vary   = True     # True: top & bottom constant, left/right varied; False: allow all configurable
All_side_const_temp_boundary = False    # if all sides constant boundary is wanted at T_bound for dataset testing

# MODEL PARAMETERS

BATCH_POINTS = 32768//2
EPOCHS = 60
LR = 1e-3
WEIGHT_DECAY = 1e-4# first case 0.0
VAL_SAMPLES = 4
VAL_BATCH_POINTS = 32768//2
STEPS_PER_EPOCH = 64

#each path weight
alpha_add=1.0
alpha_prod=0.2

DEEPONET ARCHITECTURE STARTS HERE -->

In [ ]:
import glob
# -----------------------------
# 0) Config / Auto-run directory
# -----------------------------
ROOT = Path(".")
# hardcoded one: RUN_DIR = Path("./dataset_run_YYYYMMDD-HHMMSS")
def _latest_run_dir(root: Path) -> Path | None:
    cands = sorted([Path(p) for p in glob.glob(str(root / "dataset_run_*")) if Path(p).is_dir()])
    return cands[-1] if cands else None

RUN_DIR = _latest_run_dir(ROOT)
assert RUN_DIR is not None, "No dataset_run_* folder found. Build dataset first."

DATA_TEMP = RUN_DIR / "deeponet_temp_dataset.npz"
DATA_SPLIT = RUN_DIR / "splits.json"

print(f"Using RUN_DIR: {RUN_DIR}")

In [ ]:
# -----------------------------
# 1) Load dataset + splits
# -----------------------------
D = np.load(DATA_TEMP, allow_pickle=True)
branch_Q   = D["branch_Q"]            # [C, S_Q_all]
branch_BC  = D["branch_BC"]           # [C, S_BC]
trunk_xy   = D["trunk_xy"]            # [M, 2]
trunk_t    = D["trunk_t"]             # [M, 1]
yT         = D["yT"].reshape(-1, 1)   # [M, 1]
case_ids   = D["case_ids"].astype(np.int64)  # [M]
meta       = D["meta"].item()

N         = int(meta["N"])
Lx, Ly    = float(meta["Lx"]), float(meta["Ly"])
save_times= np.array(meta["save_times"], dtype=float)
S_BC      = int(meta.get("S_BC", branch_BC.shape[1]))

print(f"Loaded: branch_Q{branch_Q.shape}, branch_BC{branch_BC.shape}, "
      f"trunk_xy{trunk_xy.shape}, trunk_t{trunk_t.shape}, yT{yT.shape}, cases={branch_Q.shape[0]}")

if DATA_SPLIT.exists():
    splits = json.loads(Path(DATA_SPLIT).read_text())
    train_ids = np.array(splits["train"], dtype=int)
    val_ids   = np.array(splits["val"], dtype=int)
    test_ids  = np.array(splits["test"], dtype=int)
else:
    C = branch_Q.shape[0]
    idx = np.arange(C); np.random.default_rng(2024).shuffle(idx)
    n_tr = int(round(0.8*C)); n_val = int(round(0.1*C))
    train_ids = idx[:n_tr]; val_ids = idx[n_tr:n_tr+n_val]; test_ids = idx[n_tr+n_val:]
    splits = {"train": train_ids.tolist(), "val": val_ids.tolist(), "test": test_ids.tolist()}
    Path(DATA_SPLIT).write_text(json.dumps(splits, indent=2))
    print("splits.json not found → wrote default 80/10/10 split.")

print(f"Split sizes → train:{len(train_ids)}  val:{len(val_ids)}  test:{len(test_ids)}")


In [ ]:
print("S_Q_all:", branch_Q.shape[1], "| S_BC:", S_BC)
print("Example rows:",
      "bQ:", branch_Q[0,:5],
      "bBC:", branch_BC[0,:5],
      "xy:", trunk_xy[:2],
      "t:", trunk_t[:2].ravel())

In [ ]:
# -----------------------------
# 2) Build samplers / normalization (TRAIN ONLY stats)
# -----------------------------
def _mask_from_cases(allowed_case_ids: np.ndarray) -> np.ndarray:
    allowed = np.zeros(branch_Q.shape[0], dtype=bool)
    allowed[allowed_case_ids] = True
    return allowed[case_ids]  # per-row mask on pooled points

mask_tr = _mask_from_cases(train_ids)
mask_val= _mask_from_cases(val_ids)
mask_te = _mask_from_cases(test_ids)

xy_tr = trunk_xy[mask_tr]       # [M_tr, 2]
t_tr  = trunk_t[mask_tr]        # [M_tr, 1]
y_tr  = yT[mask_tr]
y_mean = float(y_tr.mean())
y_std  = float(y_tr.std() + 1e-6)
cid_tr= case_ids[mask_tr]

xy_val = trunk_xy[mask_val]
t_val  = trunk_t[mask_val]
y_val  = yT[mask_val]
cid_val= case_ids[mask_val]

# branch stats (train cases only) — SEPARATE means/stds for Q and BC
bQ_tr   = branch_Q[train_ids]    # [C_tr, S_Q_all]
bBC_tr  = branch_BC[train_ids]   # [C_tr, S_BC]
bQ_mean = bQ_tr.mean(axis=0, keepdims=True); bQ_std  = bQ_tr.std(axis=0, keepdims=True) + 1e-8
bBC_mean= bBC_tr.mean(axis=0, keepdims=True); bBC_std = bBC_tr.std(axis=0, keepdims=True) + 1e-8

# coordinate/time normalization
xy_min = np.array([0.0, 0.0], dtype=np.float32)
xy_max = np.array([Lx, Ly], dtype=np.float32)
t_min  = float(save_times.min()); t_max = float(save_times.max())

def norm_y(y):
    return ((y - y_mean) / y_std).astype(np.float32)

def denorm_y(y_hat):
    # y_hat is torch tensor; returns torch tensor (Kelvin)
    return y_hat * y_std + y_mean

def norm_branch_Q(bQ):
    return ((bQ - bQ_mean) / bQ_std).astype(np.float32)

def norm_branch_BC(bBC):
    return ((bBC - bBC_mean) / bBC_std).astype(np.float32)

def norm_xy(xy):
    return ((xy - xy_min) / np.maximum(xy_max - xy_min, 1e-6)).astype(np.float32)

def norm_t(tt):
    return ((tt - t_min) / max(t_max - t_min, 1e-6)).astype(np.float32)


In [ ]:
# -----------------------------
# 3) Dataset for pooled point sampling (4-stream)
# -----------------------------
class PooledPointDataset4(Dataset):
    def __init__(self, trunk_xy, trunk_t, yT, case_ids,
                 branch_Q, branch_BC, allowed_case_ids,
                 batch_points=65536):
        self.xy = trunk_xy
        self.tt = trunk_t
        self.y  = yT.astype(np.float32)
        self.case_ids = case_ids
        self.branch_Q = branch_Q
        self.branch_BC= branch_BC

        self.allowed = np.zeros(self.branch_Q.shape[0], dtype=bool)
        self.allowed[allowed_case_ids] = True
        self.rows = np.where(self.allowed[self.case_ids])[0]   # indices eligible
        self.batch_points = int(batch_points)

        # pre-norm trunk
        self.xy_n = norm_xy(self.xy)
        self.tt_n = norm_t(self.tt)

    def __len__(self):
        return 10_000_000  # virtual

    def __getitem__(self, idx):
        ridx = np.random.randint(0, self.rows.shape[0], size=(self.batch_points,))
        rows = self.rows[ridx]

        xy   = self.xy_n[rows]           # [B,2]
        tt   = self.tt_n[rows]           # [B,1]
        y    = self.y[rows]              # [B,1]
        y_n  = norm_y(y)
        cids = self.case_ids[rows]
        bQ   = norm_branch_Q(self.branch_Q[cids])   # [B, S_Q_all]
        bBC  = norm_branch_BC(self.branch_BC[cids]) # [B, S_BC]

        return (torch.from_numpy(bQ),
                torch.from_numpy(bBC),
                torch.from_numpy(xy),
                torch.from_numpy(tt),
                torch.from_numpy(y_n))


In [ ]:
# -----------------------------
# 4) Model: 4-network DeepONet with dot-product head
# -----------------------------
class MLP(nn.Module):
    def __init__(self, in_dim, hidden, out_dim, act=nn.GELU, dropout=0.3):
        super().__init__()
        layers = []
        dims = (in_dim,) + tuple(hidden) + (out_dim,)
        for i in range(len(dims)-2):
            layers += [nn.Linear(dims[i], dims[i+1]), act(), nn.Dropout(dropout)]
        layers += [nn.Linear(dims[-2], dims[-1])]
        self.net = nn.Sequential(*layers)
    def forward(self, x): return self.net(x)

class DeepONet4(nn.Module):
    """
    y(x,y,t | Q, BC) = ⟨ BQ(Q), BBC(BC), TXY(x,y), TT(t) ⟩_⊙  + b
                      = sum( BQ ⊙ BBC ⊙ TXY ⊙ TT ) + b
    Where each subnetwork outputs a D-dim embedding (shared D).
    """
    def __init__(self, S_Q_all, S_BC,
                 D=256,                       # 256
                 q_hidden=(256,256,256),          #,(256,256,256),
                 bc_hidden=(256,256,256),         #(256,256,256),
                 xy_hidden=(256,256,256,256),     #(256,256,256,256),
                 t_hidden=(128,128)):           #(128,128)
        super().__init__()
        self.branchQ  = MLP(S_Q_all, q_hidden, D)
        self.branchBC = MLP(S_BC,    bc_hidden, D)
        self.trunkXY  = MLP(2,       xy_hidden, D)
        self.trunkT   = MLP(1,       t_hidden,  D)

        # normalizers
        self.lnQ  = nn.LayerNorm(D)
        self.lnBC = nn.LayerNorm(D)
        self.lnXY = nn.LayerNorm(D)
        self.lnT  = nn.LayerNorm(D)

        self.scale4 = (D ** 0.5)

        # NEW: expressive head on concatenation
        self.head = MLP(in_dim=4*D, hidden=(256,128), out_dim=1, act=nn.GELU)
        # tiny learned scalars to weight each path
        self.alpha_concat = nn.Parameter(torch.tensor(alpha_add))
        self.alpha_prod   = nn.Parameter(torch.tensor(alpha_prod))
        self.bias         = nn.Parameter(torch.zeros(1))

    def forward(self, bQ, bBC, xy, tt):
        eQ  = self.lnQ(self.branchQ(bQ))
        eBC = self.lnBC(self.branchBC(bBC))
        eXY = self.lnXY(self.trunkXY(xy))
        eT  = self.lnT(self.trunkT(tt))

        # expressive additive path
        z_concat = torch.cat([eQ, eBC, eXY, eT], dim=1)  # [B, 4D]
        y_concat = self.head(z_concat)                   # [B, 1]

        # multiplicative residual path (scaled)
        y_prod = (eQ * eBC * eXY * eT).sum(dim=1, keepdim=True) / self.scale4

        # combine
        yhat_norm = self.alpha_concat * y_concat + self.alpha_prod * y_prod + self.bias
        return yhat_norm

In [ ]:
import torch, importlib
print(torch.__version__)
importlib.import_module("torch._utils")
import torch.optim as optim
optim.Adam([torch.nn.Parameter(torch.randn(2,requires_grad=True))], lr=1e-3)
print("Adam OK")

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device used: {device}")
print("\n")


In [ ]:
# -----------------------------
# 5) Train / Validate
# -----------------------------

S_Q_all = branch_Q.shape[1]
model = DeepONet4(S_Q_all=S_Q_all, S_BC=S_BC,
                  D=256,
                  q_hidden=(256,256,256),
                  bc_hidden=(256,256,256),
                  xy_hidden=(256,256,256,256),
                  t_hidden=(128,128)).to(device)


BATCH_POINTS = BATCH_POINTS
EPOCHS = EPOCHS
LR = LR
WEIGHT_DECAY = WEIGHT_DECAY
VAL_SAMPLES = VAL_SAMPLES
VAL_BATCH_POINTS = VAL_BATCH_POINTS
STEPS_PER_EPOCH = STEPS_PER_EPOCH


ds_tr = PooledPointDataset4(trunk_xy=xy_tr, trunk_t=t_tr, yT=y_tr, case_ids=cid_tr,
                            branch_Q=branch_Q, branch_BC=branch_BC, allowed_case_ids=train_ids,
                            batch_points=BATCH_POINTS)
ds_va = PooledPointDataset4(trunk_xy=xy_val, trunk_t=t_val, yT=y_val, case_ids=cid_val,
                            branch_Q=branch_Q, branch_BC=branch_BC, allowed_case_ids=val_ids,
                            batch_points=VAL_BATCH_POINTS)
ds_tr.batch_points = BATCH_POINTS
ds_va.batch_points = BATCH_POINTS

opt = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)  # safer defaults
best_val = math.inf
ckpt_path = RUN_DIR / "deeponet_temp_model_4net.pt"

print("Starting training…")
for epoch in range(1, EPOCHS+1):
    model.train()
    train_loss_acc = 0.0
    train_rmseK_acc = 0.0

    for _ in range(STEPS_PER_EPOCH):
        bQ, bBC, xy, tt, ytrue_n = ds_tr[0]
        bQ=bQ.to(device); bBC=bBC.to(device); xy=xy.to(device); tt=tt.to(device); ytrue_n=ytrue_n.to(device)

        opt.zero_grad(set_to_none=True)
        yhat_n = model(bQ, bBC, xy, tt)
        loss = ((yhat_n - ytrue_n)**2).mean()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()

        with torch.no_grad():
            # report RMSE(K) on this mini-batch
            yhat_K = denorm_y(yhat_n)
            y_K    = denorm_y(ytrue_n)
            train_rmseK_acc += torch.sqrt(((yhat_K - y_K)**2).mean()).item()
            train_loss_acc  += loss.item()

    # validation over a few random batches
    model.eval()
    with torch.no_grad():
        vloss_acc = 0.0
        rmseK_acc = 0.0
        for _ in range(VAL_SAMPLES):
            bQv, bBCv, xyv, ttv, yv_n = ds_va[0]
            bQv=bQv.to(device); bBCv=bBCv.to(device); xyv=xyv.to(device); ttv=ttv.to(device); yv_n=yv_n.to(device)
            yhatv_n = model(bQv, bBCv, xyv, ttv)
            vloss_acc += ((yhatv_n - yv_n)**2).mean().item()
            yhatv_K = denorm_y(yhatv_n); yv_K = denorm_y(yv_n)
            rmseK_acc += torch.sqrt(((yhatv_K - yv_K)**2).mean()).item()

    train_mse_n  = train_loss_acc / STEPS_PER_EPOCH
    train_rmse_K = train_rmseK_acc / STEPS_PER_EPOCH
    vloss        = vloss_acc / VAL_SAMPLES
    val_rmseK    = rmseK_acc / VAL_SAMPLES

    print(f"alphas → concat={model.alpha_concat.item():.4f} | prod={model.alpha_prod.item():.4f}")

    print(f"Epoch {epoch:03d} | train MSE(n){train_mse_n:.6e} | val MSE(n){vloss:.6e} "
          f"| train RMSE(K){train_rmse_K:.3f} | val RMSE(K){val_rmseK:.3f}")

    if vloss < best_val:
        best_val = vloss
        torch.save({
            "model": model.state_dict(),
            "bQ_mean": bQ_mean, "bQ_std": bQ_std,
            "bBC_mean": bBC_mean, "bBC_std": bBC_std,
            "xy_min": xy_min, "xy_max": xy_max,
            "t_min": t_min, "t_max": t_max,
            "y_mean": y_mean, "y_std": y_std,
            "S_Q_all": branch_Q.shape[1], "S_BC": S_BC,
            "meta": meta,
        }, ckpt_path)

print(f"Best val MSE(n): {best_val:.6e}")
print(f"Saved checkpoint → {ckpt_path}")


In [ ]:
#-------------------------------
# 6) INFERENCE PLOTS
#-------------------------------

@torch.no_grad()
def load_trained(path: Path):
    ck = torch.load(path, map_location="cpu",weights_only=False)
    # Try to read minimal shape config; fall back to defaults if missing
    S_Q_all = ck.get("S_Q_all", ck.get("S_Q", None))
    S_BC    = ck.get("S_BC", None)
    assert S_Q_all is not None and S_BC is not None, "Checkpoint missing S_Q_all/S_BC"

    # Instantiate DeepONet4 with defaults used at train time
    m = DeepONet4(S_Q_all=S_Q_all, S_BC=S_BC).to(device)
    m.load_state_dict(ck["model"], strict=True)
    m.eval()

    # Stats for normalization / denormalization
    stats = {
        "bQ_mean": ck["bQ_mean"], "bQ_std": ck["bQ_std"],
        "bBC_mean": ck["bBC_mean"], "bBC_std": ck["bBC_std"],
        "xy_min": ck["xy_min"], "xy_max": ck["xy_max"],
        "t_min": ck["t_min"], "t_max": ck["t_max"],
        "y_mean": ck.get("y_mean", 0.0), "y_std": ck.get("y_std", 1.0),
        "S_Q_all": S_Q_all, "S_BC": S_BC,
        "meta": ck["meta"],
    }
    return m, stats

def _norm_branch_separate(stats, bQ_case, bBC_case):
    bQn  = (bQ_case[None, :]  - stats["bQ_mean"])  / (stats["bQ_std"]  + 1e-8)  # [1, S_Q_all]
    bBCn = (bBC_case[None, :] - stats["bBC_mean"]) / (stats["bBC_std"] + 1e-8)  # [1, S_BC]
    return bQn.astype(np.float32), bBCn.astype(np.float32)

def _norm_xy_for(stats, xy):
    return ((xy - stats["xy_min"]) / np.maximum(stats["xy_max"] - stats["xy_min"], 1e-6)).astype(np.float32)

def _norm_t_for(stats, tt):
    return ((tt - stats["t_min"]) / max(stats["t_max"] - stats["t_min"], 1e-6)).astype(np.float32)

def _denorm_y_for(stats, y_norm_tensor):
    # tensor -> Kelvin (tensor)
    return y_norm_tensor * stats["y_std"] + stats["y_mean"]

def _upsample_nn(coarse: np.ndarray, N: int) -> np.ndarray:
    """Nearest-neighbor upsample from SxS -> NxN (no external deps)."""
    S = coarse.shape[0]
    if S == N:
        return coarse.astype(np.float32)
    xi = (np.linspace(0, S-1, N)).round().astype(int)
    yi = (np.linspace(0, S-1, N)).round().astype(int)
    return coarse[np.ix_(yi, xi)].astype(np.float32)

def _reconstruct_Q_map_from_branchQ(case_id: int, meta_local: dict) -> np.ndarray:
    """
    Our branch_Q is concatenated across Nt snapshots.
    For static Q, any one block (per-time length) is fine. Use the FIRST block.
    """
    N   = int(meta_local["N"])
    Nt  = len(meta_local["save_times"])
    mode = meta_local.get("sensor_mode", "full")
    bQ_all = branch_Q[case_id]  # [S_Q_all]

    if mode == "full":
        per_time = N * N
        assert bQ_all.shape[0] % Nt == 0 and (bQ_all.shape[0] // Nt) == per_time, \
            "branch_Q does not match expected per-time length N*N."
        return bQ_all[:per_time].reshape(N, N).astype(np.float32)
    else:
        S = int(meta_local.get("S_down", 40))
        per_time = S * S
        assert bQ_all.shape[0] % Nt == 0 and (bQ_all.shape[0] // Nt) == per_time, \
            "branch_Q does not match expected per-time length S_down*S_down."
        coarse = bQ_all[:per_time].reshape(S, S)
        return _upsample_nn(coarse, N)

def _reconstruct_true_T_maps(case_id: int, meta_local: dict) -> np.ndarray:
    """
    Rebuild true T maps from branch_T (if available) or skip if not needed.
    We use the frac dataset's branch_T which stores concatenated snapshot grids.
    """
    N = int(meta_local["N"])
    times = np.array(meta_local["save_times"], dtype=float)
    Nt = len(times)
    mode = meta_local.get("sensor_mode", "full")

    Dfrac = np.load(RUN_DIR / "deeponet_frac_dataset.npz", allow_pickle=True)
    Tmat = Dfrac["branch_T"][case_id]  # [S_T = per_time * Nt]

    if mode == "full":
        return Tmat.reshape(Nt, N, N).astype(np.float32)
    else:
        S = int(meta_local.get("S_down", 40))
        coarse = Tmat.reshape(Nt, S, S)
        return np.stack([_upsample_nn(coarse[k], N) for k in range(Nt)], axis=0)

@torch.no_grad()
def predict_case_maps(ckpt: Path, case_id: int, sel_times: np.ndarray):
    model, stats = load_trained(ckpt)
    meta_local = stats["meta"]
    N   = int(meta_local["N"]); Lx = float(meta_local["Lx"]); Ly = float(meta_local["Ly"])

    # branch vecs (separate normalization)
    bQ_case  = branch_Q[case_id]    # [S_Q_all]
    bBC_case = branch_BC[case_id]   # [S_BC]
    bQn, bBCn = _norm_branch_separate(stats, bQ_case, bBC_case)  # each [1, *]
    bQ_t  = torch.from_numpy(bQn).to(device)   # [1, S_Q_all]
    bBC_t = torch.from_numpy(bBCn).to(device)  # [1, S_BC]

    # grid (N x N)
    xs = np.linspace(0.0, Lx, N, dtype=np.float32)
    ys = np.linspace(0.0, Ly, N, dtype=np.float32)
    X, Y = np.meshgrid(xs, ys, indexing="xy")
    XY = np.stack([X, Y], axis=-1).reshape(-1, 2)  # [N*N, 2]
    XY_n = _norm_xy_for(stats, XY)                 # [N*N, 2]
    XY_t = torch.from_numpy(XY_n).to(device)

    outs = []
    for t in sel_times:
        tt = np.full((XY.shape[0], 1), float(t), dtype=np.float32)  # [N*N, 1]
        tt_n = _norm_t_for(stats, tt)                               # [N*N, 1]
        tt_t = torch.from_numpy(tt_n).to(device)

        # tile branch to match grid points
        B = XY_t.shape[0]
        bQ_tile  = bQ_t.repeat(B, 1)    # [B, S_Q_all]
        bBC_tile = bBC_t.repeat(B, 1)   # [B, S_BC]

        # model outputs normalized T → denormalize
        yhat_n = model(bQ_tile, bBC_tile, XY_t, tt_t)             # [B,1] normalized
        yhat_K = _denorm_y_for(stats, yhat_n).cpu().numpy()       # [B,1] Kelvin
        outs.append(yhat_K.reshape(N, N).astype(np.float32))
    return np.stack(outs, axis=0)  # [Tsel, N, N]

def _pick_times(times_all: np.ndarray, num_cols: int = 5, prefer: np.ndarray | None = None):
    """Pick ~evenly spaced time stamps (or nearest to 'prefer' if provided)."""
    times_all = np.array(times_all, dtype=float)
    if prefer is None:
        if len(times_all) <= num_cols:
            return times_all
        idx = np.linspace(0, len(times_all)-1, num_cols).round().astype(int)
        return times_all[idx]
    # map preferred to nearest in times_all
    out = []
    for t in prefer:
        out.append(times_all[np.argmin(np.abs(times_all - t))])
    # keep unique in order
    uniq = []
    for t in out:
        if len(uniq)==0 or abs(uniq[-1]-t) > 1e-12:
            uniq.append(t)
    return np.array(uniq[:num_cols], dtype=float)

def plot_case_transient(ckpt_path: Path, case_id: int, num_cols: int = 5, prefer_times=None):
    # meta from dataset file we loaded earlier (global 'meta')
    meta_local = meta
    N   = int(meta_local["N"])
    Lx  = float(meta_local["Lx"]); Ly = float(meta_local["Ly"])
    times_all = np.array(meta_local["save_times"], dtype=float)
    sel_times = _pick_times(times_all, num_cols=num_cols, prefer=prefer_times)

    # data: Q map, true T maps, predicted T maps
    Q_map      = _reconstruct_Q_map_from_branchQ(case_id, meta_local)   # [N,N]
    T_true_all = _reconstruct_true_T_maps(case_id, meta_local)          # [Nt,N,N]
    idx_true   = np.array([np.argmin(np.abs(times_all - t)) for t in sel_times], dtype=int)
    T_true     = T_true_all[idx_true]                                   # [Tsel,N,N]
    T_pred     = predict_case_maps(ckpt_path, case_id, sel_times)       # [Tsel,N,N]

    # consistent color scaling for T
    vmin = min(T_true.min(), T_pred.min())
    vmax = max(T_true.max(), T_pred.max())

    cols = len(sel_times)
    fig, axes = plt.subplots(nrows=3, ncols=cols, figsize=(3.2*cols, 9.0), constrained_layout=True)

    # Row 0: Heat source (repeat same Q for alignment)
    for c in range(cols):
        ax = axes[0, c]
        im = ax.imshow(Q_map.T, origin="lower", extent=[0, Lx, 0, Ly], cmap="RdBu_r")
        ax.set_xticks([]); ax.set_yticks([])
        if c == 0: ax.set_ylabel("Heat source Q", fontsize=11)
        ax.set_title(f"t = {sel_times[c]:g}s", fontsize=11)
    fig.colorbar(im, ax=axes[0, :].ravel().tolist(), fraction=0.02, pad=0.02)

    # Row 1: True T
    for c in range(cols):
        ax = axes[1, c]
        imT = ax.imshow(T_true[c].T, origin="lower", extent=[0, Lx, 0, Ly],cmap="inferno", vmin=vmin, vmax=vmax,aspect="equal")
        ax.set_xticks([]); ax.set_yticks([])
        if c == 0: ax.set_ylabel("True T", fontsize=11)
    fig.colorbar(imT, ax=axes[1, :].ravel().tolist(), fraction=0.02, pad=0.02).set_label("T[K]")

    # Row 2: Pred T
    for c in range(cols):
        ax = axes[2, c]
        imP = ax.imshow(T_pred[c].T, origin="lower", extent=[0, Lx, 0, Ly], cmap="inferno",vmin=vmin, vmax=vmax, aspect="equal")
        ax.set_xticks([]); ax.set_yticks([])
        if c == 0: ax.set_ylabel("Pred T", fontsize=11)
    fig.colorbar(imP, ax=axes[2, :].ravel().tolist(), fraction=0.02, pad=0.02).set_label("T[K]")

    plt.show()


In [ ]:
# Pick a demo case and plot
ckpt_path = Path(r'runs/dataset_run_20251112-182232/deeponet_temp_model_4net.pt')

for i in range(len(test_ids)):
    demo_case = int(test_ids[i]) # if len(test_ids) else val_ids[i])
    plot_case_transient(ckpt_path, demo_case, num_cols=7,prefer_times=save_times)
    print(demo_case+1) # 0-based indexing